In [ ]:
"""
Семинар: Основы обработки биомедицинских сигналов и изображений
Цель: Показать применение NumPy и SciPy для решения реальных задач.

Часть 1: Анализ электрокардиограммы (ЭКГ) — фильтрация шумов и расчет частоты сердечных сокращений.
Часть 2: Анализ медицинских изображений — бинаризация и автоматический подсчет клеток.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy import ndimage

print("="*75)
print("БИОМЕДИЦИНСКАЯ ИНЖЕНЕРИЯ: СИГНАЛЫ И ИЗОБРАЖЕНИЯ")
print("="*75)

# =====================================================================
# ЧАСТЬ 1: АНАЛИЗ ЭКГ (СИГНАЛЫ)
# =====================================================================

def generate_synthetic_ecg(fs=200, duration=10):
    """Генерация реалистичного зашумленного сигнала ЭКГ (спрятано от студентов)."""
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    # Базовый ритм (пульс около 72 уд/мин)
    ecg_clean = signal.sawtooth(2 * np.pi * 1.2 * t, 0.1) 
    ecg_clean = np.where(ecg_clean > 0.8, ecg_clean ** 3, 0) # Острые R-пики
    # Шум 1: Изолиния "плавает" (дыхание пациента, низкочастотный шум)
    baseline_wander = 1.5 * np.sin(2 * np.pi * 0.2 * t)
    # Шум 2: Наводка от электросети (50 Гц)
    powerline_noise = 0.5 * np.sin(2 * np.pi * 50 * t)
    # Шум 3: Белый шум (мышечные сокращения)
    muscle_noise = np.random.normal(0, 0.1, len(t))
    
    ecg_noisy = ecg_clean + baseline_wander + powerline_noise + muscle_noise
    return t, ecg_noisy

print("\n--- ЧАСТЬ 1: ОБРАБОТКА ЭКГ ---")
fs = 200 # Частота дискретизации (Гц), 200 измерений в секунду
t, ecg_raw = generate_synthetic_ecg(fs=fs, duration=10)

# Шаг 1.1: Визуализация сырого сигнала
plt.figure(figsize=(12, 4))
plt.plot(t, ecg_raw, color='gray', alpha=0.7)
plt.title("Сырая ЭКГ (Зашумленная)")
plt.xlabel("Время (с)")
plt.ylabel("Амплитуда (мВ)")
plt.grid(True)
plt.show()

# Шаг 1.2: Фильтрация сигнала
print("Задание 1: Очистите сигнал от шумов.")
# Чтобы убрать плавающую изолинию (низкие частоты) и наводку 50 Гц, 
# применим полосовой фильтр (Bandpass filter), пропускающий частоты от 0.5 до 40 Гц.

# TODO: Создайте фильтр Баттерворта 4-го порядка.
# Используйте функцию signal.butter. 
# Аргументы: N=4 (порядок), Wn=[0.5, 40] (границы в Гц), btype='bandpass', fs=fs (частота)
# Функция возвращает коэффициенты b, a
# b, a = signal.butter(...)

# TODO: Примените фильтр к ecg_raw с помощью функции signal.filtfilt (она не сдвигает фазу).
# ecg_filtered = signal.filtfilt(...)

# ЗАГЛУШКА (удалить после выполнения TODO)
ecg_filtered = ecg_raw 

plt.figure(figsize=(12, 4))
plt.plot(t, ecg_filtered, color='blue')
plt.title("Очищенная ЭКГ")
plt.grid(True)
plt.show()

# Шаг 1.3: Поиск пульса (R-пиков)
print("Задание 2: Найдите удары сердца (R-пики).")

# TODO: Используйте signal.find_peaks для поиска пиков на ecg_filtered.
# Установите параметр height=0.5 (ищем только высокие пики) 
# и distance=fs*0.4 (минимальное расстояние между пиками - 0.4 секунды, чтобы не ловить двойные зубцы).
# Функция возвращает индексы пиков (peaks) и словарь свойств (удалите заглушку).
# peaks, _ = signal.find_peaks(...)

peaks = np.array([200, 400, 600]) # ЗАГЛУШКА

# Шаг 1.4: Расчет пульса (ЧСС)
# TODO: Рассчитайте расстояния между соседними пиками в индексах, используя np.diff
# rr_intervals_idx = ...

# TODO: Переведите эти расстояния в секунды (разделите на частоту дискретизации fs)
# rr_intervals_sec = ...

# TODO: Переведите секунды в пульс (удары в минуту: 60 / время_в_секундах)
# heart_rates = ...

# TODO: Выведите средний пульс пациента с помощью np.mean()
# print(f"Средний пульс пациента: {np.mean(heart_rates):.1f} уд/мин")

# Визуализация найденных пиков
plt.figure(figsize=(12, 4))
plt.plot(t, ecg_filtered, color='blue', label='ЭКГ')
plt.plot(t[peaks], ecg_filtered[peaks], 'rx', markersize=10, label='R-пики')
plt.title("Детектирование ударов сердца")
plt.legend()
plt.grid(True)
plt.show()


# =====================================================================
# ЧАСТЬ 2: АНАЛИЗ ИЗОБРАЖЕНИЙ (МАТРИЦЫ)
# =====================================================================

def generate_synthetic_cells(size=512, num_cells=40):
    """Генерация снимка флуоресцентной микроскопии клеток."""
    np.random.seed(42)
    img = np.zeros((size, size))
    # Случайные координаты клеток
    x = np.random.randint(20, size-20, num_cells)
    y = np.random.randint(20, size-20, num_cells)
    img[x, y] = 1.0
    # Размываем точки, превращая их в "пятна" клеток
    img = ndimage.gaussian_filter(img, sigma=5)
    # Нормализуем яркость от 0 до 255
    img = (img / img.max()) * 200 
    # Добавляем фоновый шум матрицы камеры
    img += np.random.normal(20, 10, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)

print("\n--- ЧАСТЬ 2: ОБРАБОТКА МЕДИЦИНСКИХ ИЗОБРАЖЕНИЙ ---")
img_cells = generate_synthetic_cells()

# Шаг 2.1: Изучение матрицы
print("Задание 3: Изучите структуру изображения.")
# TODO: Выведите размерность матрицы img_cells (атрибут .shape)
# print(f"Размерность снимка: ... пикселей")

# TODO: Выведите максимальное и минимальное значение яркости пикселя (методы .max() и .min())
# print(f"Яркость от ... до ...")

# Покажем оригинальный снимок
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img_cells, cmap='gray')
ax[0].set_title("Сырой снимок клеток")
ax[0].axis('off')

# Шаг 2.2: Гистограмма яркости
# Чтобы отделить клетки от фона, нужно посмотреть распределение пикселей.
# TODO: Сделайте массив img_cells плоским (одномерным) с помощью метода .flatten()
# TODO: Постройте гистограмму на ax[1] с помощью ax[1].hist(плоский_массив, bins=50, color='gray')
ax[1].set_title("Гистограмма яркости")
plt.show()

# Шаг 2.3: Бинаризация (Thresholding)
print("Задание 4: Отделите клетки от фона (бинаризация).")
# Глядя на гистограмму, выберите порог (threshold). Всё что ярче порога — клетка, что темнее — фон.
# TODO: Создайте бинарную маску. Просто напишите условие: img_cells > ВАШ_ПОРОГ (например, 100)
# mask = ...

mask = np.zeros_like(img_cells) # ЗАГЛУШКА

# Шаг 2.4: Подсчет объектов (Магия SciPy)
print("Задание 5: Автоматически посчитайте количество клеток.")
# Функция ndimage.label находит изолированные группы "единичек" (наши клетки) 
# и присваивает каждой группе свой номер.
# TODO: Передайте вашу маску (mask) в функцию ndimage.label()
# Она возвращает два значения: размеченную матрицу и количество найденных объектов.
# labeled_array, num_features = ndimage.label(...)

num_features = 0 # ЗАГЛУШКА
labeled_array = mask # ЗАГЛУШКА

print(f"Анализатор нашел {num_features} клеток в кадре.")

# Визуализация результата
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(mask, cmap='gray')
ax[0].set_title("Бинарная маска (клетки = белые)")
ax[0].axis('off')

# Показываем размеченную матрицу. Параметр cmap='nipy_spectral' раскрасит каждую клетку в свой цвет
ax[1].imshow(labeled_array, cmap='nipy_spectral')
ax[1].set_title(f"Распознанные объекты (Всего: {num_features})")
ax[1].axis('off')

plt.tight_layout()
plt.show()
